# Taxa preparation

Builds the canonical taxa handoff used by the results notebooks: the publication table
(`taxa_publications.parquet`), the nested taxon matches (`taxa_matches.parquet`), the
GBIF described-species benchmark (`gbif_broad_benchmark.csv`), and a provenance/audit
manifest.

The notebook takes one of two routes, chosen automatically in the configuration cell:

- **Full rebuild** (sections 1–3) when the upstream taxa coding outputs
  (`data/coding-taxa/all/`) and the screening/coding corpus files are present.
- **Regroup** (section 4) otherwise. The grouping rules in
  `checklists/mappings/taxa_broad_groups.json` are re-applied to the committed handoff:
  every taxon item in `taxa_matches.parquet` stores its LLM name and rank, its GBIF
  match status, and its accepted GBIF lineage, which are the only inputs of the
  labelling pipeline's group assignment. This route was validated to reproduce the
  committed handoff exactly under the original rules.

Both routes read the GBIF extract built by
`python -m data_helpers.sources.build_gbif_extract` (see `data/README.md`).


In [1]:
import sys
from pathlib import Path

import pandas as pd

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data_helpers").is_dir() and (candidate / "checklists").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the biodiversity repository root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_helpers import results_config
from data_helpers.prep import corpus
from data_helpers.prep import taxa_analysis_prep


In [2]:
RESULTS_CONFIG = results_config.load_results_config()
CONFIG = RESULTS_CONFIG["taxa_analysis_prep"]
DRIVER_ORDER, _, _ = results_config.driver_l1_display(RESULTS_CONFIG)
THREAT_ORDER, _, _ = results_config.threat_l0_display(RESULTS_CONFIG)
MAPPING_PATH = PROJECT_ROOT / CONFIG["taxa_group_mapping"]
MAPPING = taxa_analysis_prep.load_taxa_mapping(MAPPING_PATH)
STORE = taxa_analysis_prep.TaxaPreparedStore(
    PROJECT_ROOT / CONFIG["output_directory"]
)


# Route selection: the full rebuild needs every upstream source.
from data_helpers._config import get_merged_corpus_config

_corpus_config = get_merged_corpus_config()
UPSTREAM_SOURCES = [
    PROJECT_ROOT / CONFIG["taxa_source"],
    PROJECT_ROOT / CONFIG["taxa_lineage_source"],
    _corpus_config["screening"]["path"],
    *(source["path"] for source in _corpus_config["coding_sources"]),
]
MISSING_UPSTREAM = [path for path in UPSTREAM_SOURCES if not Path(path).exists()]
FULL_REBUILD = not MISSING_UPSTREAM
print("Route:", "full rebuild" if FULL_REBUILD else "regroup the committed handoff (section 4)")
for path in MISSING_UPSTREAM:
    print(f"  upstream source not present: {Path(path).relative_to(PROJECT_ROOT).as_posix()}")


Route: regroup the committed handoff (section 4)
  upstream source not present: data/coding-taxa/all/taxa_all.parquet
  upstream source not present: data/coding-taxa/all/taxa_lineages.parquet
  upstream source not present: data/screened-partitions/all/screening_all.csv
  upstream source not present: data/coding-driver/all/driver_all.csv
  upstream source not present: data/coding-threats_l0/all/threats_l0_all.csv
  upstream source not present: data/coding-geography/all/geography_all.csv
  upstream source not present: data/coding-ecosystems/all/ecosystems_all.csv
  upstream source not present: data/coding-study/all/study_all.csv


## 1. Build the publication-level handoff

The configured eligible corpus already enforces one row per `UT` across the coding sources. Taxa enrichment is joined one-to-one and the original key order must remain unchanged. Driver, Threat-L0, realm, and all three taxonomic resolutions are retained as ordered list columns.

In [3]:
if FULL_REBUILD:
    corpus_df = corpus.build_merged_corpus()
    articles, audits = taxa_analysis_prep.build_taxa_publications(
        corpus_df,
        taxa_path=PROJECT_ROOT / CONFIG["taxa_source"],
        mapping=MAPPING,
        driver_order=DRIVER_ORDER,
        threat_order=THREAT_ORDER,
    )

    display(pd.DataFrame(audits["source_grain"]).style.format(
        {"rows": "{:,}", "unique_UT": "{:,}", "missing_UT": "{:,}", "duplicated_UT": "{:,}"}
    ).hide(axis="index"))
    print(f"Prepared publication grain: {len(articles):,} rows / {articles['UT'].nunique():,} unique UTs.")


In [4]:
if FULL_REBUILD:
    broad_accounting = (
        articles["taxa_broad_inclusion_state"]
        .value_counts()
        .reindex(taxa_analysis_prep.INCLUSION_ORDER, fill_value=0)
        .rename_axis("taxa_broad_inclusion_state")
        .rename("n_publications")
        .reset_index()
    )
    broad_accounting["share_of_all_publications_pct"] = broad_accounting["n_publications"] / len(articles) * 100
    display(broad_accounting.style.format({"n_publications": "{:,}", "share_of_all_publications_pct": "{:.2f}%"}).hide(axis="index"))
    display(pd.DataFrame(audits["match_status"]).style.format({"taxon_item_count": "{:,}", "share_of_taxon_items_pct": "{:.2f}%"}).hide(axis="index"))


## 2. Build the fixed GBIF broad-group benchmark

The curated GBIF backbone is streamed once. Only accepted species-rank records enter; the current broad hierarchy rules are applied, unresolved records are excluded, and benchmarkable shares are renormalized.

In [5]:
if FULL_REBUILD:
    benchmark = taxa_analysis_prep.build_gbif_broad_benchmark(
        PROJECT_ROOT / CONFIG["gbif_source"],
        MAPPING_PATH,
    )
    display(benchmark.audit.style.format({"records": "{:,}"}).hide(axis="index"))
    display(benchmark.counts.style.format({"described_species_count": "{:,}", "described_species_share_pct": "{:.2f}%"}).hide(axis="index"))


## 3. Write and verify the canonical bundle

The manifest records source signatures, the semantic grouping-rule fingerprint, row reconciliation, all compact processing audits, and benchmark construction counts. A reload immediately verifies schema, rules, key grain, and artifact row counts.

In [6]:
if FULL_REBUILD:
    match_summary = STORE.write_matches(
        PROJECT_ROOT / CONFIG["taxa_lineage_source"],
        expected_uts=articles["UT"],
    )
    sources = {
        "corpus_config": PROJECT_ROOT / CONFIG["corpus_config"],
        "taxa": PROJECT_ROOT / CONFIG["taxa_source"],
        "taxa_lineage": PROJECT_ROOT / CONFIG["taxa_lineage_source"],
        "gbif": PROJECT_ROOT / CONFIG["gbif_source"],
        "gbif_eml": PROJECT_ROOT / CONFIG["gbif_eml_source"],
        "taxa_group_mapping": MAPPING_PATH,
    }
    manifest = taxa_analysis_prep.build_manifest(
        articles, benchmark, mapping=MAPPING, audits=audits, match_summary=match_summary, sources=sources, repository_root=PROJECT_ROOT
    )
    bundle = taxa_analysis_prep.TaxaPreparedBundle(articles, benchmark, manifest)
    written = STORE.write(bundle)
    reloaded = STORE.load(mapping=MAPPING)
    assert reloaded.articles["UT"].equals(articles["UT"])
    pd.testing.assert_frame_equal(reloaded.benchmark.counts, benchmark.counts, check_exact=False, rtol=1e-12, atol=1e-12)
    print("Verified canonical bundle:")
    for path in written:
        print(f"  {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size / 1024**2:.2f} MiB)")


## 4. Regroup the committed handoff (when upstream files are absent)

Re-applies the current grouping rules item by item, rebuilds the GBIF benchmark from
the extract under the same rules, rewrites the bundle in place, and reloads it. The
manifest keeps the previous build's upstream sources (`upstream_sources`) and records
the regrouped-from artifacts and rule fingerprint (`regrouped_from`). The streaming
regroup takes roughly 15 minutes.


In [7]:
if not FULL_REBUILD:
    GBIF_PATH = PROJECT_ROOT / CONFIG["gbif_source"]
    GBIF_EML_PATH = PROJECT_ROOT / CONFIG["gbif_eml_source"]
    missing_gbif = [path for path in (GBIF_PATH, GBIF_EML_PATH) if not path.exists()]
    if missing_gbif:
        raise FileNotFoundError(
            "Build the GBIF extract first: python -m data_helpers.sources.build_gbif_extract"
        )
    written = STORE.rebuild_with_rules(
        mapping_path=MAPPING_PATH,
        gbif_path=GBIF_PATH,
        gbif_eml_path=GBIF_EML_PATH,
        repository_root=PROJECT_ROOT,
    )
    bundle = STORE.load(mapping=MAPPING)
    print("Verified regrouped bundle:")
    for path in written:
        print(f"  {path.relative_to(PROJECT_ROOT).as_posix()} ({path.stat().st_size / 1024**2:.2f} MiB)")
    display(
        pd.Series(bundle.manifest["broad_inclusion_counts"], name="n_publications")
        .rename_axis("taxa_broad_inclusion_state").reset_index()
        .style.format({"n_publications": "{:,}"}).hide(axis="index")
    )
    display(bundle.benchmark.audit.style.hide(axis="index"))
    display(
        bundle.benchmark.counts.style.format(
            {"described_species_count": "{:,}", "described_species_share_pct": "{:.3f}%"}
        ).hide(axis="index")
    )


Verified regrouped bundle:
  notebooks/data_processing/outputs/02_taxa_analysis_prep/taxa_publications.parquet (14.90 MiB)
  notebooks/data_processing/outputs/02_taxa_analysis_prep/taxa_matches.parquet (54.48 MiB)
  notebooks/data_processing/outputs/02_taxa_analysis_prep/gbif_broad_benchmark.csv (0.00 MiB)
  notebooks/data_processing/outputs/02_taxa_analysis_prep/manifest.json (0.03 MiB)


taxa_broad_inclusion_state,n_publications
Excluded: Not applicable only,"119,503"
Excluded: Unclear only,"4,501"
Excluded: accepted taxon but broad hierarchy unresolved,"4,446"
Excluded: mixed Not applicable and Unclear,0
Excluded: no taxon item returned,0
Excluded: reported taxon but no accepted benchmarkable match,"31,949"
Included: at least one benchmarkable broad group,"444,800"


metric,value
GBIF backbone rows scanned,7746724
Species-rank rows,4994322
Accepted species-rank rows,2614593
Accepted species rows missing taxonID,0
Broad-group benchmark total,2614593


broad_group,described_species_count,described_species_share,described_species_share_pct
Chordates,"120,268",0.045999,4.600%
Non-chordate animals,"1,694,587",0.648126,64.813%
Plants,"446,842",0.170903,17.090%
Fungi,"172,327",0.065910,6.591%
Other,"173,190",0.066240,6.624%
Unresolved,"7,379",0.002822,0.282%
